# 2. Discounted cash flow

**Question.** What is equity worth from the cash the business can generate under these assumptions?

**Output.** WACC, Gordon and exit-multiple prices, TV share of EV, sensitivity tables (level and Δ), and a full 3D sensitivity panel with cross-sections through the base case.

Unlevered FCF = `EBIT × (1 − t) + D&A − capex − ΔNWC`. Net debt is the latest balance sheet only.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "valuation" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from the project directory.")

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataclasses import replace
import matplotlib.pyplot as plt
import pandas as pd

from valuation import boot
from valuation import plots as viz
from valuation import report as R
from valuation.case import TOL
from valuation.statements import Line, audit, combine, forecast_statements, identity_failures, max_abs_gap

# Compact equity picker — changes cases/active.yaml for EVERY notebook.
case, UNIT = boot(picker=True)


## 1. Base-case output


In [ ]:
from valuation.dcf import run_dcf, unlevered_fcf

forecast = forecast_statements(case, interest="beginning")
result = run_dcf(case)
R.show_dcf(result, case)


## 2. Free cash flow and present value


In [ ]:
flows = unlevered_fcf(case, forecast.statements.income, forecast.statements.cashflow)
R.table(pd.DataFrame({"UFCF": flows}), digits=1, caption="Unlevered free cash flow by year")
R.table(result.gordon_schedule, digits=2, caption="Gordon discounting schedule")
R.table(result.exit_schedule, digits=2, caption="Exit-multiple discounting schedule")
viz.plot_ufcf_waterfall(result.gordon_schedule, UNIT)
plt.show()
viz.plot_dcf_pv_split(result)
plt.show()


## 3. Sensitivity — level and change versus base


In [ ]:
g_row = viz.nearest_label(result.gordon_sensitivity.index, result.wacc, as_pct=True)
g_col = viz.nearest_label(result.gordon_sensitivity.columns, case.assumptions.terminal_growth, as_pct=True)
R.table(result.gordon_sensitivity, digits=2, caption="Gordon $/share — rows WACC, columns terminal growth")
viz.plot_sensitivity_pair(result.gordon_sensitivity, "Gordon equity value per share", "$ / share", base_row=g_row, base_col=g_col)
plt.show()

e_row = viz.nearest_label(result.exit_sensitivity.index, result.wacc, as_pct=True)
e_col = viz.nearest_label(result.exit_sensitivity.columns, case.assumptions.exit_ev_ebitda, as_multiple=True)
R.table(result.exit_sensitivity, digits=2, caption="Exit-multiple $/share — rows WACC, columns exit EV/EBITDA")
viz.plot_sensitivity_pair(result.exit_sensitivity, "Exit-multiple equity value per share", "$ / share", base_row=e_row, base_col=e_col)
plt.show()


## 4. DCF scenario workspace

Choose WACC and a terminal assumption, or click the target map to set both. The selected valuation, target gap, terminal exposure and sensitivity slices update together from the full forecast DCF.

Enter a target price to solve for the WACC or terminal assumption that would reach it. Pin scenarios to compare and export them. Changing assumptions here does not overwrite the case. The operating forecast stays fixed; Gordon scenarios with WACC at or below growth remain undefined.


In [ ]:
from valuation.surface_workbench import show_dcf_workbench
show_dcf_workbench(case)
